<a href="https://colab.research.google.com/github/Balajivallepu/Nlp_projects/blob/main/NLP_project_7_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jiashenliu/515k-hotel-reviews-data-in-europe")

print("Path to dataset files:", path)

Using Colab cache for faster access to the '515k-hotel-reviews-data-in-europe' dataset.
Path to dataset files: /kaggle/input/515k-hotel-reviews-data-in-europe


In [45]:
pip install kagglehub

In [46]:
import kagglehub

path = kagglehub.dataset_download("jiashenliu/515k-hotel-reviews-data-in-europe")

print(path)

Using Colab cache for faster access to the '515k-hotel-reviews-data-in-europe' dataset.
/kaggle/input/515k-hotel-reviews-data-in-europe


In [47]:
import os

print(os.listdir(path))

['Hotel_Reviews.csv']


In [48]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('vader_lexicon')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [49]:
import pandas as pd
import os

file_path = os.path.join(path, "Hotel_Reviews.csv")

df = pd.read_csv(file_path)

df.head()

,Hotel_Address,Additional_Number_of_Scoring,Review_Date,Average_Score,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Total_Number_of_Reviews,Positive_Review,Review_Total_Positive_Word_Counts,Total_Number_of_Reviews_Reviewer_Has_Given,Reviewer_Score,Tags,days_since_review,lat,lng
0,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,8/3/2017,7.7,Hotel Arena,Russia,I am so angry that i made this post available...,397,1403,Only the park outside of the hotel was beauti...,11,7,2.9,"[' Leisure trip ', ' Couple ', ' Duplex Double...",0 days,52.360576,4.915968
1,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,8/3/2017,7.7,Hotel Arena,Ireland,No Negative,0,1403,No real complaints the hotel was great great ...,105,7,7.5,"[' Leisure trip ', ' Couple ', ' Duplex Double...",0 days,52.360576,4.915968
2,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/31/2017,7.7,Hotel Arena,Australia,Rooms are nice but for elderly a bit difficul...,42,1403,Location was good and staff were ok It is cut...,21,9,7.1,"[' Leisure trip ', ' Family with young childre...",3 days,52.360576,4.915968
3,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/31/2017,7.7,Hotel Arena,United Kingdom,My room was dirty and I was afraid to walk ba...,210,1403,Great location in nice surroundings the bar a...,26,1,3.8,"[' Leisure trip ', ' Solo traveler ', ' Duplex...",3 days,52.360576,4.915968
4,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/24/2017,7.7,Hotel Arena,New Zealand,You When I booked with your company on line y...,140,1403,Amazing location and building Romantic setting,8,3,6.7,"[' Leisure trip ', ' Couple ', ' Suite ', ' St...",10 days,52.360576,4.915968


In [50]:
print(df.shape)

(515738, 17)


In [51]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 515738 entries, 0 to 515737
Data columns (total 17 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   Hotel_Address                               515738 non-null  object 
 1   Additional_Number_of_Scoring                515738 non-null  int64  
 2   Review_Date                                 515738 non-null  object 
 3   Average_Score                               515738 non-null  float64
 4   Hotel_Name                                  515738 non-null  object 
 5   Reviewer_Nationality                        515738 non-null  object 
 6   Negative_Review                             515738 non-null  object 
 7   Review_Total_Negative_Word_Counts           515738 non-null  int64  
 8   Total_Number_of_Reviews                     515738 non-null  int64  
 9   Positive_Review                             515738 non-null  object 
 

In [52]:
print(df.isnull().sum())

Hotel_Address                                    0
Additional_Number_of_Scoring                     0
Review_Date                                      0
Average_Score                                    0
Hotel_Name                                       0
Reviewer_Nationality                             0
Negative_Review                                  0
Review_Total_Negative_Word_Counts                0
Total_Number_of_Reviews                          0
Positive_Review                                  0
Review_Total_Positive_Word_Counts                0
Total_Number_of_Reviews_Reviewer_Has_Given       0
Reviewer_Score                                   0
Tags                                             0
days_since_review                                0
lat                                           3268
lng                                           3268
dtype: int64


In [53]:
print(df.columns)

Index(['Hotel_Address', 'Additional_Number_of_Scoring', 'Review_Date',
       'Average_Score', 'Hotel_Name', 'Reviewer_Nationality',
       'Negative_Review', 'Review_Total_Negative_Word_Counts',
       'Total_Number_of_Reviews', 'Positive_Review',
       'Review_Total_Positive_Word_Counts',
       'Total_Number_of_Reviews_Reviewer_Has_Given', 'Reviewer_Score', 'Tags',
       'days_since_review', 'lat', 'lng'],
      dtype='object')


In [54]:
df = df[
    [
        "Hotel_Name",
        "Reviewer_Score",
        "Negative_Review",
        "Positive_Review"
    ]
]

In [55]:
df["Review"] = (
    df["Negative_Review"].fillna("")
    + " "
    + df["Positive_Review"].fillna("")
)

In [56]:
df = df[df["Review"].str.strip() != ""]

In [57]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):

    text = str(text).lower()

    text = re.sub(r'[^a-zA-Z ]', '', text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

df["Clean_Review"] = df["Review"].apply(preprocess)

print(df[["Review","Clean_Review"]].head())

                                              Review  \
0   I am so angry that i made this post available...   
1  No Negative  No real complaints the hotel was ...   
2   Rooms are nice but for elderly a bit difficul...   
3   My room was dirty and I was afraid to walk ba...   
4   You When I booked with your company on line y...   

                                        Clean_Review  
0  angry made post available via possible site us...  
1  negative real complaint hotel great great loca...  
2  room nice elderly bit difficult room two story...  
3  room dirty afraid walk barefoot floor looked c...  
4  booked company line showed picture room though...  


In [58]:
sia = SentimentIntensityAnalyzer()

def sentiment(review):

    score = sia.polarity_scores(review)["compound"]

    if score >= 0.05:
        return "Positive"

    elif score <= -0.05:
        return "Negative"

    else:
        return "Neutral"

df["Sentiment"] = df["Review"].apply(sentiment)

print(df[["Review","Sentiment"]].head())

                                              Review Sentiment
0   I am so angry that i made this post available...  Negative
1  No Negative  No real complaints the hotel was ...  Positive
2   Rooms are nice but for elderly a bit difficul...  Positive
3   My room was dirty and I was afraid to walk ba...  Positive
4   You When I booked with your company on line y...  Positive


In [59]:
def service_issue(review):

    review = review.lower()

    if "room" in review:
        return "Room"

    elif "staff" in review or "reception" in review:
        return "Staff"

    elif "food" in review or "restaurant" in review or "breakfast" in review:
        return "Food"

    elif "clean" in review or "dirty" in review or "housekeeping" in review:
        return "Cleanliness"

    elif "wifi" in review or "pool" in review or "gym" in review or "spa" in review:
        return "Amenities"

    else:
        return "Other"

df["Service_Issue"] = df["Review"].apply(service_issue)

print(df[["Review","Service_Issue"]].head())

                                              Review Service_Issue
0   I am so angry that i made this post available...          Room
1  No Negative  No real complaints the hotel was ...          Room
2   Rooms are nice but for elderly a bit difficul...          Room
3   My room was dirty and I was afraid to walk ba...          Room
4   You When I booked with your company on line y...          Room


In [60]:
print("Customer Satisfaction Report\n")

print("Total Reviews :", len(df))

print("\nSentiment Distribution")
print(df["Sentiment"].value_counts())

print("\nService Issues")
print(df["Service_Issue"].value_counts())

positive = (df["Sentiment"] == "Positive").mean() * 100
negative = (df["Sentiment"] == "Negative").mean() * 100
neutral = (df["Sentiment"] == "Neutral").mean() * 100

print("\nPositive Reviews :", round(positive,2), "%")
print("Negative Reviews :", round(negative,2), "%")
print("Neutral Reviews :", round(neutral,2), "%")

Customer Satisfaction Report

Total Reviews : 515679

Sentiment Distribution
Sentiment
Positive    369610
Negative    109793
Neutral      36276
Name: count, dtype: int64

Service Issues
Service_Issue
Room           276478
Staff           96838
Other           90971
Food            35339
Cleanliness      9043
Amenities        7010
Name: count, dtype: int64

Positive Reviews : 71.67 %
Negative Reviews : 21.29 %
Neutral Reviews : 7.03 %


In [61]:
sample_review = "The room was clean and spacious. Staff were friendly and breakfast was delicious."

clean_review = preprocess(sample_review)

print("Clean Review:", clean_review)
print("Sentiment:", sentiment(sample_review))
print("Service Issue:", service_issue(sample_review))

Clean Review: room clean spacious staff friendly breakfast delicious
Sentiment: Positive
Service Issue: Room
